# Plate Number Text Extraction

In [ ]:
import os
import re
import shutil
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract

# --- find the Tesseract program ---
exe = shutil.which("tesseract")
if exe is None:
    for p in (r"C:\Program Files\Tesseract-OCR\tesseract.exe",
              r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"):
        if os.path.exists(p):
            exe = p
            break
if exe is None:
    raise RuntimeError("Tesseract is not installed or not on PATH. Install it, or set exe to its full path.")
pytesseract.pytesseract.tesseract_cmd = exe
print("Using Tesseract:", exe)

# --- the images to read: (label, path, layout) ---
# layout = what kind of characters each block of the plate holds, left to right:
#   "LD" = letters block, then digits block   (ABC 1234)
#   "DL" = digits block, then letters block   (123 ABC)
#   "L" / "D" = a single letters-only / digits-only block
#   "A"  = anything (falls back to pattern fixing)
# Change the layout of any plate that is not "letters then digits".
PLATES = [
    ("Private",        "Images/pn-private.png",        "LD"),
    ("Public Utility", "Images/pn-public_utility.png", "LD"),
    ("Government",     "Images/pn-government.png",     "LD"),
    ("Diplomatic",     "Images/pn-diplomatic.png",     "LD"),
    ("Others",         "Images/pn-others.png",         "LD"),
]

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
DIGITS = "0123456789"
ALNUM = LETTERS + DIGITS
TARGET_W = 1200      # every plate is resized to this width first
TEXT_H = 100         # text height (pixels) handed to OCR

In [ ]:
def load_plate(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {path}")
    s = TARGET_W / img.shape[1]
    interp = cv2.INTER_CUBIC if s > 1 else cv2.INTER_AREA
    return cv2.resize(img, None, fx=s, fy=s, interpolation=interp)


def to_ink(img, mode="gray", clahe=False):
    """Return the smoothed gray image and a rough binary mask where text pixels are 255."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if mode == "gray" else img.min(axis=2)
    smooth = cv2.GaussianBlur(gray, (0, 0), 1.5)
    if clahe:
        smooth = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(smooth)
    _, bw = cv2.threshold(smooth, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    ink = bw if cv2.countNonZero(bw) < bw.size / 2 else cv2.bitwise_not(bw)
    return smooth, ink


def find_number_line(ink):
    H, W = ink.shape
    n, lab, stats, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)

    allc, cand = [], []
    for i in range(1, n):
        x, y, w, h, a = stats[i]
        if x <= 1 or y <= 1 or x + w >= W - 1 or y + h >= H - 1:   # touches the edge: plate frame
            continue
        if w > 0.5 * W or h > 0.6 * H:                              # too big to be a character
            continue
        allc.append((i, x, y, w, h))
        if a >= 0.0003 * H * W and h >= 0.03 * H and w >= 0.1 * h:  # not a speck, not a thin bar
            cand.append((i, x, y, w, h))

    # group candidates by row; the row with the tallest characters wins
    best = None
    for (_, x, y, w, h) in cand:
        yc = y + h / 2
        row = [c for c in cand if abs(c[2] + c[4] / 2 - yc) < 0.3 * h and 0.7 * h < c[4] < 1.4 * h]
        if len(row) < 3:
            continue
        med_h = float(np.median([c[4] for c in row]))
        if best is None or med_h > best[0]:
            best = (med_h, row)
    if best is None:
        return None

    line_h, row = best
    yc_line = float(np.median([c[2] + c[4] / 2 for c in row]))
    gx0 = min(c[1] for c in row)
    gx1 = max(c[1] + c[3] for c in row)

    members = [c for c in allc
               if abs(c[2] + c[4] / 2 - yc_line) < 0.6 * line_h
               and c[4] <= 1.5 * line_h
               and gx0 - 0.4 * line_h <= c[1] + c[3] / 2 <= gx1 + 0.4 * line_h]

    x0 = min(c[1] for c in members);        x1 = max(c[1] + c[3] for c in members)
    y0 = min(c[2] for c in members);        y1 = max(c[2] + c[4] for c in members)
    mask = np.isin(lab, [c[0] for c in members])
    return {"box": (x0, y0, x1, y1), "mask": mask}

In [ ]:
def padded_box(box, shape, m=0.08):
    """Grow the detected box a little, so strokes that were faded/cut at the edge are still inside."""
    x0, y0, x1, y1 = box
    p = max(int(m * (y1 - y0)), 2)
    H, W = shape[:2]
    return max(x0 - p, 0), max(y0 - p, 0), min(x1 + p, W), min(y1 + p, H)


def color_map(img, found, roi_box):
    """0 = plate color, 1 = text color, measured per pixel. Faded or glared strokes still score high."""
    rx0, ry0, rx1, ry1 = roi_box
    roi = img[ry0:ry1, rx0:rx1].astype(np.float32)
    old = found["mask"][ry0:ry1, rx0:rx1]
    ink_c = np.median(roi[old], axis=0)
    bg_c = np.median(roi[~old], axis=0)
    d = ink_c - bg_c
    return ((roi - bg_c) @ d) / (d @ d + 1e-6)


def ink_from_t(t, thresh, k=5):
    """Binary mask (255 = ink) from the color map; closing fills small breaks inside strokes."""
    m = (t > thresh).astype(np.uint8) * 255
    return cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))


def split_chars(mask):
    """Bounding boxes (x, y, w, h) of the character blobs, left to right."""
    H, W = mask.shape
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    boxes = []
    for i in range(1, n):
        x, y, w, h, a = stats[i]
        if x == 0 or y == 0 or x + w >= W or y + h >= H:   # touches the border: frame or junk
            continue
        if h < 0.5 * H or w > 1.5 * h:                     # dashes, dots, specks, long bars
            continue
        boxes.append((int(x), int(y), int(w), int(h)))
    return sorted(boxes, key=lambda b: b[0])


def split_groups(boxes, H):
    """Split characters into blocks (e.g. 'MNO' | '7890') at the one clearly wider gap."""
    if not boxes:
        return []
    if len(boxes) < 2:
        return [boxes]
    gaps = [boxes[i + 1][0] - (boxes[i][0] + boxes[i][2]) for i in range(len(boxes) - 1)]
    i = int(np.argmax(gaps))
    others = [g for j, g in enumerate(gaps) if j != i]
    ref = max(others) if others else 0
    if gaps[i] >= max(2 * ref, 0.25 * H):
        return [boxes[:i + 1], boxes[i + 1:]]
    return [boxes]

In [ ]:
TO_LETTER = {"0": "O", "1": "I", "2": "Z", "5": "S", "6": "G", "8": "B", "4": "A", "7": "T"}
TO_DIGIT  = {"O": "0", "D": "0", "Q": "0", "I": "1", "L": "1", "Z": "2",
             "S": "5", "G": "6", "B": "8", "A": "4", "T": "7"}

# only used when a plate cannot be split into the blocks its layout expects
TEMPLATES = ["LLLDDDD", "LLLDDD", "DDDLLL", "DDDDLL"]


def prepare(dark_on_white, x0, x1):
    """Crop columns x0:x1, add a white border, scale to TEXT_H tall, light blur."""
    H = dark_on_white.shape[0]
    crop = dark_on_white[:, x0:x1]
    pad = int(0.35 * H)
    crop = cv2.copyMakeBorder(crop, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=255)
    crop = cv2.resize(crop, None, fx=TEXT_H / H, fy=TEXT_H / H, interpolation=cv2.INTER_CUBIC)
    return cv2.GaussianBlur(crop, (0, 0), 0.5)


def ocr_text(im, kind, psm):
    wl = {"L": LETTERS, "D": DIGITS, "A": ALNUM}[kind]       # letters-only / digits-only / both
    cfg = (f"--oem 3 --psm {psm} -c tessedit_char_whitelist={wl} "
           f"-c load_system_dawg=0 -c load_freq_dawg=0")
    return re.sub(r"[^A-Z0-9]", "", pytesseract.image_to_string(im, config=cfg).upper())


def read_group(variants, kind, n_expected):
    """OCR every image variant, then vote. Answers with the expected character count are preferred."""
    votes = Counter()
    for im in variants:
        for psm in (7, 8):
            t = ocr_text(im, kind, psm)
            if t:
                votes[t] += 1
    if not votes:
        return "", votes
    pool = {t: c for t, c in votes.items() if len(t) == n_expected} or dict(votes)
    return max(pool, key=lambda t: pool[t]), votes


def force_kind(text, kind):
    if kind == "L":
        return "".join(TO_LETTER.get(c, c) for c in text)
    if kind == "D":
        return "".join(TO_DIGIT.get(c, c) for c in text)
    return text


def fix_characters(text):
    """Fallback: force each character to letter/digit using the plate pattern that needs the fewest changes."""
    s = re.sub(r"[^A-Z0-9]", "", text.upper())
    best, best_cost = None, None
    for t in TEMPLATES:
        if len(t) != len(s):
            continue
        out, cost = [], 0
        for ch, kind in zip(s, t):
            if kind == "L" and ch.isdigit():
                ch, cost = TO_LETTER.get(ch, ch), cost + 1
            elif kind == "D" and ch.isalpha():
                ch, cost = TO_DIGIT.get(ch, ch), cost + 1
            out.append(ch)
        if best is None or cost < best_cost:
            best, best_cost = "".join(out), cost
    result = best if best else s
    return re.sub(r"(?<=[A-Z])(?=\d)|(?<=\d)(?=[A-Z])", " ", result)

In [ ]:
def read_plate(label, path, layout="LD", show=True):
    img = load_plate(path)

    # 1) find the big number line (try a few ways of separating text from plate)
    found = None
    for mode, clahe in (("gray", True), ("gray", False), ("min", False)):
        smooth, ink = to_ink(img, mode, clahe)
        found = find_number_line(ink)
        if found is not None:
            break
    if found is None:
        print(f"{label}: no number line found")
        return ""

    # 2) rebuild the text mask by color distance (recovers faded / glared strokes)
    roi_box = padded_box(found["box"], img.shape)
    t = color_map(img, found, roi_box)
    base = ink_from_t(t, 0.35)
    boxes = split_chars(base)
    groups = split_groups(boxes, base.shape[0])

    # 3) several versions of the same line: 3 thresholds + a soft grayscale version
    variants = [255 - ink_from_t(t, th) for th in (0.25, 0.35, 0.45)]
    variants.append((255 * (1 - np.clip(t, 0, 1))).astype(np.uint8))
    W = base.shape[1]

    # 4) read each block with the right character set (letters block -> letters only, etc.)
    if len(groups) == len(layout):
        parts = []
        for g, kind in zip(groups, layout):
            gx0 = max(min(b[0] for b in g) - 4, 0)
            gx1 = min(max(b[0] + b[2] for b in g) + 4, W)
            txt, _ = read_group([prepare(v, gx0, gx1) for v in variants], kind, len(g))
            parts.append(force_kind(txt, kind))
        text = " ".join(parts)
        note = ""
    else:
        txt, _ = read_group([prepare(v, 0, W) for v in variants], "A", len(boxes))
        text = fix_characters(txt)
        note = f"   (found {len(groups)} block(s), layout '{layout}' expects {len(layout)}: used fallback)"

    if show:
        rx0, ry0, rx1, ry1 = roi_box
        boxed = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        cv2.rectangle(boxed, (rx0, ry0), (rx1, ry1), (255, 0, 0), 4)

        seg = cv2.cvtColor(255 - base, cv2.COLOR_GRAY2RGB)
        for (x, y, w, h) in boxes:
            cv2.rectangle(seg, (x, y), (x + w, y + h), (0, 160, 0), 2)

        fig, axs = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw={"width_ratios": [2, 2, 1]})
        axs[0].imshow(boxed); axs[0].set_title(f"{label}: detected number line"); axs[0].axis("off")
        axs[1].imshow(seg); axs[1].set_title("Text mask + characters found"); axs[1].axis("off")
        axs[2].plot((base > 0).sum(axis=1), np.arange(base.shape[0])); axs[2].invert_yaxis()
        axs[2].set_title("Ink per row"); axs[2].set_xlabel("pixels")
        plt.tight_layout(); plt.show()

    print(f"{label}: Extracted Text: {text}{note}")
    return text


results = [(label, read_plate(label, path, layout)) for label, path, layout in PLATES]

print("\n==== SUMMARY ====")
for label, text in results:
    print(f"{label:<15}: {text}")